# 🧠 Strategist Agent — AI Career Rejection Analyst

**Pipeline Stage: Skills Gap Analysis → Match Scoring → 30-Day Roadmap**

This notebook takes a **parsed Resume (JSON)** and a **parsed Job Description (JSON)**, then:
1. Extracts skills from both documents
2. Computes a **weighted match score** (0–100)
3. Identifies the **skills gap** (missing skills by priority)
4. Generates a **30-day learning roadmap** via Mistral AI

---

## 📦 Step 1: Setup & Dependencies

In [26]:
!pip install mistralai python-dotenv -q

import json
import os
import re
from dotenv import load_dotenv
from mistralai import Mistral

print("✅ All dependencies installed and imported.")

✅ All dependencies installed and imported.


## 🔑 Step 2: Mistral Client Setup
Loads `MISTRAL_API_KEY` from the project `.env` file and initializes the client.

In [27]:
# Load environment variables from the project root .env file
load_dotenv(dotenv_path=os.path.join("..", "..", ".env"))

MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "").strip()

if not MISTRAL_API_KEY:
    raise ValueError(
        "❌ MISTRAL_API_KEY not found! "
        "Add it to your .env file: MISTRAL_API_KEY=your_key_here"
    )

client = Mistral(api_key=MISTRAL_API_KEY)
MODEL = "mistral-small-latest"

print(f"✅ Mistral client initialized (key: {MISTRAL_API_KEY[:6]}...{MISTRAL_API_KEY[-4:]})")
print(f"   Model: {MODEL}")

✅ Mistral client initialized (key: kNVCjm...YmSX)
   Model: mistral-small-latest


## 📄 Step 3: Mock Resume Data (Parsed JSON)
This is the structured resume output from the earlier OCR → Parser stage.

In [28]:
mock_resume = {
    "name": "John Doe",
    "title": "Data Scientist",
    "sections": {
        "summary": {
            "paragraphs": [
                "Highly motivated and experienced Data Scientist with a strong background in machine learning, data analysis, and database administration. Proven track record of delivering high-quality projects and driving business growth through data-driven insights."
            ]
        },
        "experience": {
            "subsections": {
                "senior_data_scientist": {
                    "bullet_points": [
                        "Jan 2020 - Present",
                        "New York, USA",
                        "Developed and deployed machine learning models using Python, R, and SQL",
                        "Designed and implemented data pipelines using Apache Beam, Apache Spark, and AWS Glue",
                        "Collaborated with data engineers to optimize database performance and improve data storage"
                    ],
                    "paragraphs": [
                        "### ABC Corporation",
                        "Led a team of data scientists to develop and implement predictive models that increased sales by 25% and reduced costs by 15%. Collaborated with cross-functional teams to design and deploy data pipelines, resulting in a 30% improvement in data quality and a 40% reduction in data processing time."
                    ]
                },
                "data_scientist": {
                    "bullet_points": [
                        "Jun 2018 - Dec 2019",
                        "San Francisco, USA",
                        "Developed and deployed machine learning models using Python, R, and SQL",
                        "Collaborated with data engineers to design and implement data pipelines",
                        "Worked with business stakeholders to identify opportunities and develop data-driven solutions"
                    ],
                    "paragraphs": [
                        "### DEF Startups",
                        "Worked on multiple projects, including customer segmentation, churn prediction, and recommendation systems. Developed and deployed models that resulted in a 20% increase in customer engagement and a 15% reduction in customer churn."
                    ]
                }
            }
        },
        "projects": {
            "subsections": {
                "customer_segmentation": {
                    "bullet_points": [
                        "Mar 2020 - Jun 2020",
                        "Python", "R", "SQL", "Apache Spark"
                    ],
                    "paragraphs": [
                        "Lead Data Scientist",
                        "Developed a customer segmentation model using clustering algorithms that resulted in a 25% increase in targeted marketing campaigns."
                    ]
                },
                "churn_prediction": {
                    "bullet_points": [
                        "Sep 2019 - Dec 2019",
                        "Python", "R", "SQL", "Apache Spark"
                    ],
                    "paragraphs": [
                        "Data Scientist",
                        "Developed a churn prediction model using machine learning algorithms that resulted in a 20% reduction in customer churn."
                    ]
                }
            }
        },
        "skills": {
            "bullet_points": [
                "Machine Learning", "Data Analysis", "Database Administration",
                "Python", "R", "SQL", "Apache Spark",
                "Communication", "Team Management"
            ]
        },
        "languages": {
            "bullet_points": ["English", "Spanish"]
        },
        "education": {
            "subsections": {
                "master's": {
                    "bullet_points": ["Jun 2018", "Stanford, USA"],
                    "paragraphs": ["Stanford University", "Data Science • 3.8"]
                }
            }
        },
        "certificates": {
            "subsections": {
                "certified_data_scientist": {
                    "bullet_points": ["Dec 2019"],
                    "paragraphs": [
                        "Data Science Council of America",
                        "Certified data scientist with expertise in machine learning, data analysis, and database administration."
                    ]
                }
            }
        },
        "strengths": {
            "subsections": {
                "strong_analytical_and_problem_solving_skills": {
                    "paragraphs": ["Able to analyze complex data sets and develop creative solutions to drive business growth."]
                },
                "excellent_communication_and_teamwork_skills": {
                    "paragraphs": ["Able to effectively communicate technical concepts to non-technical stakeholders and collaborate with cross-functional teams to drive project success."]
                }
            }
        }
    }
}

print(f"✅ Resume loaded: {mock_resume['name']} — {mock_resume['title']}")
print(f"   Listed skills: {', '.join(mock_resume['sections']['skills']['bullet_points'])}")

✅ Resume loaded: John Doe — Data Scientist
   Listed skills: Machine Learning, Data Analysis, Database Administration, Python, R, SQL, Apache Spark, Communication, Team Management


## 📋 Step 4: Job Description Input (Raw Text)
Paste a **real job description** below — copy-paste it straight from **LinkedIn**, **Indeed**,
or any other job board. The Mistral LLM will automatically extract:
- Job title, company, location
- Required, preferred, and nice-to-have skills
- Responsibilities, experience requirements, education

No manual formatting needed — just paste the text as-is.

In [29]:
# ════════════════════════════════════════════════════════════
# PASTE YOUR JOB DESCRIPTION BELOW (raw text from LinkedIn/Indeed)
# ════════════════════════════════════════════════════════════

raw_jd_text = """
Senior Machine Learning Engineer
TechVision AI · New York, USA (Hybrid)

About the job

We are seeking a Senior ML Engineer to design, build, and deploy
production-grade machine learning systems. You will work closely with data
scientists and platform engineers to bring models from prototype to
production at scale.

Responsibilities
• Design and implement scalable ML pipelines for training and inference
• Deploy models to production using Docker, Kubernetes, and cloud platforms
• Build monitoring and alerting systems for model performance drift
• Collaborate with data scientists to optimize model architectures
• Implement A/B testing frameworks for model evaluation
• Maintain ML infrastructure and CI/CD pipelines

Requirements
• 5+ years in ML / Data Science, 2+ years in ML Engineering
• MS or PhD in Computer Science, Data Science, or related field
• Strong proficiency in Python, SQL
• Hands-on experience with Machine Learning, Deep Learning, PyTorch
• Experience with Docker, Kubernetes, MLflow
• Knowledge of CI/CD Pipelines and REST APIs

Preferred Qualifications
• Experience with Apache Spark, TensorFlow
• AWS experience (SageMaker, Lambda, S3)
• Terraform for infrastructure as code
• Data Analysis skills
• LLM Fine-tuning experience

Nice to Have
• Rust or Go programming
• Graph Neural Networks experience
• Team Management / leadership
"""

print(f"✅ Job description loaded ({len(raw_jd_text.strip())} characters)")
print(f"   Preview: {raw_jd_text.strip()[:80]}...")


✅ Job description loaded (1360 characters)
   Preview: Senior Machine Learning Engineer
TechVision AI · New York, USA (Hybrid)

About t...


In [30]:
JD_PARSE_SYSTEM_PROMPT = """You are an expert job description parser.
Read the raw job posting text and extract ALL information into a clean,
structured JSON object.

RULES:
1. Extract the job title, company name, and location from the posting.
2. Write a clear description summarizing the role.
3. Classify EVERY skill/technology mentioned into exactly ONE of these tiers:
   - required_skills: skills explicitly listed as required, mandatory, or must-have
   - preferred_skills: skills listed as preferred, nice-to-have-but-important, or bonus
   - nice_to_have: skills listed as optional, nice-to-have, or "a plus"
   If the posting does not clearly separate tiers, use your judgment based on how
   the skill is described (e.g. "strong proficiency" → required, "experience with" → preferred).
4. Extract responsibilities as a list of strings.
5. Extract experience requirements as a single string.
6. Extract education requirements as a single string.
7. Return ONLY valid JSON — no markdown fences, no explanation, no extra text.

OUTPUT FORMAT (follow this exactly):
{
  "job_title": "...",
  "company": "...",
  "location": "...",
  "description": "...",
  "requirements": {
    "required_skills": ["skill1", "skill2", ...],
    "preferred_skills": ["skill1", "skill2", ...],
    "nice_to_have": ["skill1", "skill2", ...]
  },
  "experience_required": "...",
  "education": "...",
  "responsibilities": ["...", "..."]
}
"""


def parse_jd_with_llm(raw_text: str) -> dict:
    """
    Send raw job description text to Mistral and get back structured JSON.
    """
    print("🤖 Parsing job description with Mistral AI...")
    print(f"   Model: {MODEL}")
    print(f"   Input: {len(raw_text.strip())} characters")
    print()

    response = client.chat.complete(
        model=MODEL,
        messages=[
            {"role": "system", "content": JD_PARSE_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Parse this job description:\n\n{raw_text}"}
        ],
        temperature=0.1,   
        max_tokens=4096,
    )

    raw_response = response.choices[0].message.content.strip()

    cleaned = re.sub(r"^```[a-zA-Z]*\s*", "", raw_response, flags=re.MULTILINE)
    cleaned = re.sub(r"\s*```$", "", cleaned, flags=re.MULTILINE).strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"❌ Failed to parse LLM response as JSON: {e}")
        print(f"   Raw response (first 500 chars): {raw_response[:500]}")
        raise

    return parsed


# ── Parse the raw JD text ──
mock_jd = parse_jd_with_llm(raw_jd_text)

# ── Display results ──
print("=" * 60)
print("📋 PARSED JOB DESCRIPTION")
print("=" * 60)
print(f"\n🏢 {mock_jd.get('job_title', 'N/A')} @ {mock_jd.get('company', 'N/A')}")
print(f"📍 {mock_jd.get('location', 'N/A')}")
print(f"\n📝 Description:")
print(f"   {mock_jd.get('description', 'N/A')[:120]}...")

reqs = mock_jd.get("requirements", {})
print(f"\n   Required skills:  {len(reqs.get('required_skills', []))}")
for s in reqs.get("required_skills", []):
    print(f"     • {s}")
print(f"   Preferred skills: {len(reqs.get('preferred_skills', []))}")
for s in reqs.get("preferred_skills", []):
    print(f"     • {s}")
print(f"   Nice-to-have:     {len(reqs.get('nice_to_have', []))}")
for s in reqs.get("nice_to_have", []):
    print(f"     • {s}")

print(f"\n🎓 Education: {mock_jd.get('education', 'N/A')}")
print(f"📅 Experience: {mock_jd.get('experience_required', 'N/A')}")
print(f"\n✅ Job description parsed successfully!")


🤖 Parsing job description with Mistral AI...
   Model: mistral-small-latest
   Input: 1360 characters

📋 PARSED JOB DESCRIPTION

🏢 Senior Machine Learning Engineer @ TechVision AI
📍 New York, USA (Hybrid)

📝 Description:
   A Senior ML Engineer role focused on designing, building, and deploying production-grade machine learning systems. The r...

   Required skills:  10
     • Python
     • SQL
     • Machine Learning
     • Deep Learning
     • PyTorch
     • Docker
     • Kubernetes
     • MLflow
     • CI/CD Pipelines
     • REST APIs
   Preferred skills: 6
     • Apache Spark
     • TensorFlow
     • AWS (SageMaker, Lambda, S3)
     • Terraform
     • Data Analysis
     • LLM Fine-tuning
   Nice-to-have:     5
     • Rust
     • Go
     • Graph Neural Networks
     • Team Management
     • Leadership

🎓 Education: MS or PhD in Computer Science, Data Science, or related field
📅 Experience: 5+ years in ML / Data Science, 2+ years in ML Engineering

✅ Job description parsed successful

## 🔍 Step 5: Skills Extraction
Extract all candidate skills from the resume (skills section + experience + projects)  
and all JD requirements organized by priority.

In [31]:
SKILL_EXTRACT_SYSTEM_PROMPT = """You are an expert technical recruiter.
Your task is to extract ALL skills, technologies, tools, and relevant domain
knowledge from a candidate\'s resume JSON.

RULES:
1. Look everywhere: explicit skills sections, experience bullets, and projects.
2. Extract both hard skills (e.g., Python, AWS, Machine Learning) and relevant
   soft skills/domain expertise (e.g., Team Management, A/B Testing, Agile).
3. Normalize the skills: return them as a clean list of strings in Title Case.
4. Remove duplicates.
5. Return ONLY a valid JSON object containing a single array called "skills".

OUTPUT FORMAT:
{
  "skills": ["Python", "Machine Learning", "AWS", ...]
}
"""


def normalize(skill: str) -> str:
    """Lowercase and strip whitespace for flexible comparison later."""
    return skill.strip().lower()


def extract_candidate_skills_llm(resume: dict) -> set:
    """
    Ask Mistral to extract all skills from the candidate profile dynamically.
    Returns a set of normalized (lowercase) skill strings.
    """
    print("🧑 Extracting candidate skills dynamically with Mistral AI...")
    print(f"   Model: {MODEL}")
    
    # Convert the resume structure to a string for the prompt
    resume_text = json.dumps(resume, indent=2)
    
    response = client.chat.complete(
        model=MODEL,
        messages=[
            {"role": "system", "content": SKILL_EXTRACT_SYSTEM_PROMPT},
            {"role": "user",   "content": f"Extract all skills from this resume:\n\n{resume_text}"}
        ],
        temperature=0.1,
        max_tokens=2048,
    )

    raw_response = response.choices[0].message.content.strip()

    # Clean markdown fences
    cleaned = re.sub(r"^```[a-zA-Z]*\s*", "", raw_response, flags=re.MULTILINE)
    cleaned = re.sub(r"\s*```$", "", cleaned, flags=re.MULTILINE).strip()

    try:
        parsed = json.loads(cleaned)
        skill_list = parsed.get("skills", [])
    except json.JSONDecodeError as e:
        print(f"❌ Failed to parse LLM skill extraction as JSON: {e}")
        print(f"   Raw response: {raw_response[:500]}")
        # Fallback to an empty set if extraction fails
        return set()

    # Normalize to lowercase for the matching logic in Step 6
    return {normalize(s) for s in skill_list}


def extract_jd_requirements(jd: dict) -> dict:
    """
    Extract JD skills organized by priority tier, keeping exact matching format.
",
    Returns: { \'required\': set, \'preferred\': set, \'nice_to_have\': set }
    """
    reqs = jd.get("requirements", {})
    return {
        "required":     {normalize(s) for s in reqs.get("required_skills", [])},
        "preferred":    {normalize(s) for s in reqs.get("preferred_skills", [])},
        "nice_to_have": {normalize(s) for s in reqs.get("nice_to_have", [])}
    }


# ── Run extraction ──
candidate_skills = extract_candidate_skills_llm(mock_resume)
jd_requirements  = extract_jd_requirements(mock_jd)

print("\n" + "="*60)
print("📊 SKILLS EXTRACTION RESULTS")
print("="*60)
print(f"\n🧑 Candidate skills found ({len(candidate_skills)}):")
for s in sorted(candidate_skills):
    print(f"   • {s.title()}")

for s in sorted(jd_requirements["required"]):
    print(f"   • {s.title()}")

for s in sorted(jd_requirements["preferred"]):
    print(f"   • {s.title()}")

for s in sorted(jd_requirements["nice_to_have"]):
    print(f"   • {s.title()}")


🧑 Extracting candidate skills dynamically with Mistral AI...
   Model: mistral-small-latest

📊 SKILLS EXTRACTION RESULTS

🧑 Candidate skills found (25):
   • Agile
   • Apache Beam
   • Apache Spark
   • Aws Glue
   • Business Stakeholders
   • Churn Prediction
   • Clustering Algorithms
   • Communication
   • Cross-Functional Teams
   • Customer Segmentation
   • Data Analysis
   • Data Pipelines
   • Data Processing
   • Data Quality
   • Data-Driven Solutions
   • Database Administration
   • Machine Learning
   • Predictive Models
   • Problem Solving
   • Python
   • R
   • Recommendation Systems
   • Sql
   • Targeted Marketing Campaigns
   • Team Management
   • Ci/Cd Pipelines
   • Deep Learning
   • Docker
   • Kubernetes
   • Machine Learning
   • Mlflow
   • Python
   • Pytorch
   • Rest Apis
   • Sql
   • Apache Spark
   • Aws (Sagemaker, Lambda, S3)
   • Data Analysis
   • Llm Fine-Tuning
   • Tensorflow
   • Terraform
   • Go
   • Graph Neural Networks
   • Leadership
  

## ⚖️ Step 6: Weighted Match Score Calculation

**Scoring weights:**
| Tier | Weight | Reasoning |
|------|--------|-----------|
| Required | 1.0 | Must-have — makes or breaks the application |
| Preferred | 0.5 | Strong differentiator |
| Nice-to-Have | 0.25 | Bonus points |

In [32]:
# ─── SCORING WEIGHTS ───
WEIGHTS = {
    "required":     1.00,
    "preferred":    0.50,
    "nice_to_have": 0.25
}


def fuzzy_match(candidate_skill: str, jd_skill: str) -> bool:
    """
    Flexible skill matching:
      - Exact match after normalization
      - Substring containment (e.g., 'aws glue' matches 'aws (sagemaker, lambda, s3)')
      - Core keyword overlap
    """
    a, b = normalize(candidate_skill), normalize(jd_skill)
    if a == b:
        return True
    if a in b or b in a:
        return True
    # Check if any significant word (3+ chars) from candidate appears in the JD skill
    a_words = {w for w in a.split() if len(w) >= 3}
    b_words = {w for w in b.split() if len(w) >= 3}
    if a_words and b_words and a_words & b_words:
        return True
    return False


def calculate_match_score(candidate_skills: set, jd_requirements: dict) -> dict:
    """
    Calculate the weighted match score.

    Returns:
        {
            'overall_score': float (0-100),
            'max_possible_score': float,
            'raw_weighted_score': float,
            'breakdown': {
                'required':  {'matched': [...], 'missing': [...], 'weight': 1.0, 'pct': ...},
                'preferred': {'matched': [...], 'missing': [...], 'weight': 0.5, 'pct': ...},
                'nice_to_have': {'matched': [...], 'missing': [...], 'weight': 0.25, 'pct': ...}
            },
            'all_missing': [...]
        }
    """
    breakdown = {}
    total_weighted = 0
    max_weighted = 0
    all_missing = []

    for tier, jd_skills in jd_requirements.items():
        weight = WEIGHTS[tier]
        matched = []
        missing = []

        for jd_skill in sorted(jd_skills):
            found = any(fuzzy_match(cs, jd_skill) for cs in candidate_skills)
            if found:
                matched.append(jd_skill)
            else:
                missing.append(jd_skill)

        tier_max = len(jd_skills) * weight
        tier_score = len(matched) * weight
        tier_pct = (len(matched) / len(jd_skills) * 100) if jd_skills else 0

        total_weighted += tier_score
        max_weighted += tier_max

        breakdown[tier] = {
            "matched": matched,
            "missing": missing,
            "weight": weight,
            "matched_count": len(matched),
            "total_count": len(jd_skills),
            "tier_pct": round(tier_pct, 1)
        }

        all_missing.extend([(s, tier, weight) for s in missing])

    overall = (total_weighted / max_weighted * 100) if max_weighted else 0

    return {
        "overall_score": round(overall, 1),
        "max_possible_score": max_weighted,
        "raw_weighted_score": total_weighted,
        "breakdown": breakdown,
        "all_missing": all_missing
    }


# ── Calculate ──
score_result = calculate_match_score(candidate_skills, jd_requirements)

# ── Display results ──
print("="*60)
print("⚖️  WEIGHTED MATCH SCORE")
print("="*60)

# Overall score with a visual bar
score = score_result['overall_score']
bar_len = 30
filled = int(bar_len * score / 100)
bar = '█' * filled + '░' * (bar_len - filled)
print(f"\n   [{bar}] {score}%\n")

# Breakdown by tier
for tier_name, tier_data in score_result['breakdown'].items():
    label = tier_name.replace('_', ' ').title()
    m, t = tier_data['matched_count'], tier_data['total_count']
    print(f"\n── {label} (weight {tier_data['weight']}) ─ {m}/{t} matched ({tier_data['tier_pct']}%)")
    if tier_data['matched']:
        print(f"   ✅ Matched: {', '.join(s.title() for s in tier_data['matched'])}")
    if tier_data['missing']:
        print(f"   ❌ Missing: {', '.join(s.title() for s in tier_data['missing'])}")

print(f"\n{'='*60}")
print(f"📊 Total weighted: {score_result['raw_weighted_score']:.2f} / {score_result['max_possible_score']:.2f}")

⚖️  WEIGHTED MATCH SCORE

   [██████████████████████████░░░░] 89.5%


── Required (weight 1.0) ─ 9/10 matched (90.0%)
   ✅ Matched: Ci/Cd Pipelines, Deep Learning, Docker, Kubernetes, Machine Learning, Python, Pytorch, Rest Apis, Sql
   ❌ Missing: Mlflow

── Preferred (weight 0.5) ─ 5/6 matched (83.3%)
   ✅ Matched: Apache Spark, Aws (Sagemaker, Lambda, S3), Data Analysis, Tensorflow, Terraform
   ❌ Missing: Llm Fine-Tuning

── Nice To Have (weight 0.25) ─ 5/5 matched (100.0%)
   ✅ Matched: Go, Graph Neural Networks, Leadership, Rust, Team Management

📊 Total weighted: 12.75 / 14.25


## 🤖 Step 7: Mistral AI — Gap Analysis & 30-Day Roadmap
Sends the skills gap to Mistral and receives a structured learning roadmap.

In [33]:
def build_roadmap_prompt(resume: dict, jd: dict, score_result: dict) -> str:
    """
    Build the user prompt that provides all context to Mistral
    for generating the 30-day learning roadmap.
    """
    # Organize missing skills by priority
    missing_by_tier = {}
    for skill, tier, weight in score_result["all_missing"]:
        missing_by_tier.setdefault(tier, []).append(skill.title())

    missing_section = ""
    for tier in ["required", "preferred", "nice_to_have"]:
        if tier in missing_by_tier:
            label = tier.replace("_", " ").title()
            skills_list = ", ".join(missing_by_tier[tier])
            missing_section += f"  - {label}: {skills_list}\n"

    matched_skills = []
    for tier_data in score_result["breakdown"].values():
        matched_skills.extend(s.title() for s in tier_data["matched"])

    prompt = f"""## Candidate Profile
- **Name**: {resume['name']}
- **Current Title**: {resume['title']}
- **Existing Skills**: {', '.join(matched_skills)}
- **Match Score**: {score_result['overall_score']}%

## Target Role
- **Position**: {jd['job_title']}
- **Company**: {jd['company']}

## Skills Gap (Missing Skills)
{missing_section}
## Task
Based on the candidate's existing skills and the missing skills above,
generate a **detailed 30-day learning roadmap** to close the gap.

Structure your response EXACTLY as follows:

### 📊 Gap Analysis Summary
A brief analysis of the candidate's strengths vs. the role's requirements.

### 🗓️ 30-Day Learning Roadmap

#### Week 1: [Theme]
- **Day 1-2**: [Topic] — [specific activities, resources]
- **Day 3-5**: [Topic] — [specific activities, resources]
- **Day 6-7**: [Hands-on project or review]

#### Week 2: [Theme]
(same format)

#### Week 3: [Theme]
(same format)

#### Week 4: [Theme]
(same format)

### 🎯 Milestones & Success Metrics
- End of Week 1: [measurable outcome]
- End of Week 2: [measurable outcome]
- End of Week 3: [measurable outcome]
- End of Week 4: [measurable outcome]

### 📚 Recommended Resources
Top courses, books, and tools for each missing skill.

### ⏱️ Daily Time Commitment
Suggested hours per day and optimal schedule.
"""
    return prompt


SYSTEM_PROMPT = """You are an expert AI Career Strategist. Your specialty is analyzing
skills gaps between candidates and job descriptions, then creating actionable,
realistic learning roadmaps. You prioritize REQUIRED skills first, then PREFERRED,
then NICE-TO-HAVE. Your roadmaps are practical, include specific free/paid resources,
and have clear milestones. Be encouraging but realistic about timelines."""


def generate_roadmap(client, model, resume, jd, score_result):
    """
    Call Mistral to generate the 30-day learning roadmap.
    """
    user_prompt = build_roadmap_prompt(resume, jd, score_result)

    print("🤖 Sending gap analysis to Mistral AI...")
    print(f"   Model: {model}")
    print(f"   Missing skills to address: {len(score_result['all_missing'])}")
    print()

    response = client.chat.complete(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=4096
    )

    roadmap = response.choices[0].message.content
    print("✅ Roadmap generated successfully!")
    return roadmap


# ── Generate the roadmap ──
roadmap_output = generate_roadmap(client, MODEL, mock_resume, mock_jd, score_result)

🤖 Sending gap analysis to Mistral AI...
   Model: mistral-small-latest
   Missing skills to address: 2

✅ Roadmap generated successfully!


## 🚀 Step 8: Full Pipeline Output
Complete formatted results from the Strategist Agent.

In [34]:
from IPython.display import display, Markdown

# ════════════════════════════════════════════════════════════
# PIPELINE SUMMARY
# ════════════════════════════════════════════════════════════

summary_md = f"""
# 🧠 Strategist Agent — Pipeline Results

---

## 🧑 Candidate Profile
| Field | Value |
|-------|-------|
| **Name** | {mock_resume['name']} |
| **Title** | {mock_resume['title']} |
| **Education** | Stanford University — M.S. Data Science |
| **Experience** | ~6 years (Data Scientist → Senior Data Scientist) |

## 🎯 Target Role
| Field | Value |
|-------|-------|
| **Position** | {mock_jd['job_title']} |
| **Company** | {mock_jd['company']} |
| **Location** | {mock_jd['location']} |

## ⚖️ Match Score: **{score_result['overall_score']}%**

| Tier | Matched | Total | Coverage | Weight |
|------|---------|-------|----------|--------|
"""

for tier_name, tier_data in score_result['breakdown'].items():
    label = tier_name.replace('_', ' ').title()
    m = tier_data['matched_count']
    t = tier_data['total_count']
    pct = tier_data['tier_pct']
    w = tier_data['weight']
    summary_md += f"| {label} | {m} | {t} | {pct}% | ×{w} |\n"

summary_md += f"""
### ✅ Skills Matched
"""
for tier_name, tier_data in score_result['breakdown'].items():
    if tier_data['matched']:
        label = tier_name.replace('_', ' ').title()
        skills = ', '.join(f'`{s.title()}`' for s in tier_data['matched'])
        summary_md += f"- **{label}**: {skills}\n"

summary_md += f"""
### ❌ Skills Gap
"""
for tier_name, tier_data in score_result['breakdown'].items():
    if tier_data['missing']:
        label = tier_name.replace('_', ' ').title()
        skills = ', '.join(f'`{s.title()}`' for s in tier_data['missing'])
        summary_md += f"- **{label}**: {skills}\n"

summary_md += """
---

## 🗺️ AI-Generated 30-Day Learning Roadmap

"""

summary_md += roadmap_output

# ── Render as rich Markdown ──
display(Markdown(summary_md))


# 🧠 Strategist Agent — Pipeline Results

---

## 🧑 Candidate Profile
| Field | Value |
|-------|-------|
| **Name** | John Doe |
| **Title** | Data Scientist |
| **Education** | Stanford University — M.S. Data Science |
| **Experience** | ~6 years (Data Scientist → Senior Data Scientist) |

## 🎯 Target Role
| Field | Value |
|-------|-------|
| **Position** | Senior Machine Learning Engineer |
| **Company** | TechVision AI |
| **Location** | New York, USA (Hybrid) |

## ⚖️ Match Score: **89.5%**

| Tier | Matched | Total | Coverage | Weight |
|------|---------|-------|----------|--------|
| Required | 9 | 10 | 90.0% | ×1.0 |
| Preferred | 5 | 6 | 83.3% | ×0.5 |
| Nice To Have | 5 | 5 | 100.0% | ×0.25 |

### ✅ Skills Matched
- **Required**: `Ci/Cd Pipelines`, `Deep Learning`, `Docker`, `Kubernetes`, `Machine Learning`, `Python`, `Pytorch`, `Rest Apis`, `Sql`
- **Preferred**: `Apache Spark`, `Aws (Sagemaker, Lambda, S3)`, `Data Analysis`, `Tensorflow`, `Terraform`
- **Nice To Have**: `Go`, `Graph Neural Networks`, `Leadership`, `Rust`, `Team Management`

### ❌ Skills Gap
- **Required**: `Mlflow`
- **Preferred**: `Llm Fine-Tuning`

---

## 🗺️ AI-Generated 30-Day Learning Roadmap

### 📊 Gap Analysis Summary
John Doe has a strong background in machine learning, data analysis, and cloud computing, with extensive experience in Python, TensorFlow, PyTorch, and AWS services. His current skills align well with the Senior Machine Learning Engineer role at TechVision AI, scoring an impressive 89.5% match. The key gaps are:
- **Required**: MLflow for experiment tracking and model management.
- **Preferred**: LLM (Large Language Model) fine-tuning to enhance his expertise in advanced NLP tasks.

Given his existing skills, John is well-positioned to quickly upskill in these areas.

---

### 🗓️ 30-Day Learning Roadmap

#### Week 1: MLflow Fundamentals
- **Day 1-2**: Introduction to MLflow — [Read MLflow documentation](https://mlflow.org/docs/latest/index.html), complete the [MLflow Quickstart Guide](https://mlflow.org/docs/latest/quickstart.html).
- **Day 3-5**: Experiment Tracking & Model Registry — Follow the [MLflow Experiment Tracking Tutorial](https://mlflow.org/docs/latest/tracking.html) and [Model Registry Tutorial](https://mlflow.org/docs/latest/model-registry.html).
- **Day 6-7**: Hands-on Project — Implement MLflow in a personal project (e.g., tracking experiments for a simple ML model).

#### Week 2: Advanced MLflow & Deployment
- **Day 8-10**: MLflow Projects & Deployment — Learn how to package and deploy ML models using [MLflow Projects](https://mlflow.org/docs/latest/projects.html) and [MLflow Models](https://mlflow.org/docs/latest/models.html).
- **Day 11-12**: Integrate MLflow with AWS — Use [MLflow with SageMaker](https://mlflow.org/docs/latest/integrations/aws.html) and [MLflow with Lambda](https://aws.amazon.com/lambda/).
- **Day 13-14**: Hands-on Project — Deploy a model using MLflow and AWS, track experiments, and manage the model lifecycle.

#### Week 3: Introduction to LLM Fine-Tuning
- **Day 15-17**: Basics of LLMs & Fine-Tuning — Read [Hugging Face’s guide to fine-tuning](https://huggingface.co/docs/transformers/training) and complete the [Hugging Face Course](https://huggingface.co/course/chapter1/1).
- **Day 18-20**: Fine-Tuning Techniques — Explore [PEFT (Parameter-Efficient Fine-Tuning)](https://huggingface.co/docs/peft/index) and [LoRA (Low-Rank Adaptation)](https://huggingface.co/docs/transformers/main_classes/peft#lora).
- **Day 21-22**: Hands-on Project — Fine-tune a small LLM (e.g., DistilBERT) on a custom dataset using Hugging Face Transformers.

#### Week 4: Advanced LLM Fine-Tuning & Integration
- **Day 23-25**: Advanced Fine-Tuning & Evaluation — Learn about [evaluation metrics for LLMs](https://huggingface.co/docs/transformers/evaluation) and [advanced fine-tuning techniques](https://huggingface.co/docs/transformers/training#advanced-training).
- **Day 26-27**: Deploy Fine-Tuned Models — Deploy a fine-tuned model using [Hugging Face Inference API](https://huggingface.co/inference-api) or [AWS SageMaker](https://aws.amazon.com/sagemaker/).
- **Day 28-30**: Hands-on Project — Fine-tune and deploy a model, then document the process and results.

---

### 🎯 Milestones & Success Metrics
- **End of Week 1**: Successfully track experiments and log models using MLflow.
- **End of Week 2**: Deploy a model using MLflow and AWS, with a working pipeline.
- **End of Week 3**: Fine-tune a small LLM on a custom dataset and evaluate its performance.
- **End of Week 4**: Deploy a fine-tuned LLM and document the entire process.

---

### 📚 Recommended Resources
- **MLflow**:
  - [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
  - [MLflow Quickstart Guide](https://mlflow.org/docs/latest/quickstart.html)
  - [MLflow with AWS](https://mlflow.org/docs/latest/integrations/aws.html)
- **LLM Fine-Tuning**:
  - [Hugging Face Course](https://huggingface.co/course/chapter1/1)
  - [Hugging Face Transformers Documentation](https://huggingface.co/docs/transformers/index)
  - [PEFT Documentation](https://huggingface.co/docs/peft/index)

---

### ⏱️ Daily Time Commitment
- **Suggested Hours**: 2-3 hours per day.
- **Optimal Schedule**: 1-2 hours in the evening (e.g., 7-9 PM) or during weekends for longer sessions.
- **Total Time**: ~60-90 hours over 30 days.

John, you’re already well-prepared for this role! With focused effort, you can close these gaps and be fully ready for the Senior Machine Learning Engineer position at TechVision AI. Keep up the great work! 🚀

In [35]:
# ════════════════════════════════════════════════════════════
# EXPORT: Save complete results as JSON for downstream agents
# ════════════════════════════════════════════════════════════

strategist_output = {
    "agent": "strategist",
    "candidate": {
        "name": mock_resume["name"],
        "title": mock_resume["title"],
        "skills_found": sorted(list(candidate_skills))
    },
    "target_role": {
        "title": mock_jd["job_title"],
        "company": mock_jd["company"]
    },
    "match_score": {
        "overall": score_result["overall_score"],
        "breakdown": {
            tier: {
                "matched": data["matched"],
                "missing": data["missing"],
                "coverage_pct": data["tier_pct"]
            }
            for tier, data in score_result["breakdown"].items()
        }
    },
    "roadmap_markdown": roadmap_output
}

output_path = "strategist_output.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(strategist_output, f, indent=2, ensure_ascii=False)

print(f"\n💾 Results exported to: {output_path}")
print(f"   File size: {os.path.getsize(output_path):,} bytes")
print("\n✅ Strategist Agent pipeline complete!")


💾 Results exported to: strategist_output.json
   File size: 6,403 bytes

✅ Strategist Agent pipeline complete!


## 📊 Step 9: Generate Visualizations & PDF Report
Generate professional charts inline (donut gauge, bar charts, radar chart)
and compile everything into a multi-page PDF report.

In [36]:
!pip install fpdf2 matplotlib numpy -q

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import tempfile
import math
import re as re_mod
from datetime import datetime
from fpdf import FPDF
from IPython.display import display, Image as IPImage, Markdown as IPMarkdown

# ─── COLOR PALETTE ───
COLORS = {
    "primary":    "#1E3A5F",
    "accent":     "#4A90D9",
    "success":    "#27AE60",
    "warning":    "#F39C12",
    "danger":     "#E74C3C",
    "light_gray": "#F5F5F5",
    "mid_gray":   "#BDC3C7",
    "dark_gray":  "#2C3E50",
    "white":      "#FFFFFF",
    "tier_required":    "#E74C3C",
    "tier_preferred":   "#F39C12",
    "tier_nice":        "#3498DB",
}

def hex_to_rgb(hex_color):
    h = hex_color.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def hex_to_float_rgb(hex_color):
    r, g, b = hex_to_rgb(hex_color)
    return (r / 255, g / 255, b / 255)

print("✅ Visualization dependencies loaded.")

✅ Visualization dependencies loaded.


In [37]:
# ════════════════════════════════════════════════════════════
# GENERATE & DISPLAY CHARTS INLINE
# ════════════════════════════════════════════════════════════

tmp_dir = tempfile.mkdtemp(prefix="career_report_")
score = score_result['overall_score']
breakdown = score_result['breakdown']

# ── 1. Donut / Gauge Chart ──
fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(aspect="equal"))
fig.patch.set_alpha(0)

if score >= 80:
    donut_color = COLORS["success"]
elif score >= 60:
    donut_color = COLORS["warning"]
else:
    donut_color = COLORS["danger"]

sizes = [score, 100 - score]
colors_donut = [hex_to_float_rgb(donut_color), hex_to_float_rgb(COLORS["light_gray"])]
wedges, _ = ax.pie(sizes, colors=colors_donut, startangle=90, counterclock=False,
                   wedgeprops=dict(width=0.35, edgecolor="white", linewidth=2))
ax.text(0, 0.05, f"{score}%", ha="center", va="center",
        fontsize=36, fontweight="bold", color=hex_to_float_rgb(COLORS["dark_gray"]))
ax.text(0, -0.2, "Match Score", ha="center", va="center",
        fontsize=12, color=hex_to_float_rgb(COLORS["mid_gray"]))
ax.set_title("Overall Match Score", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
donut_path = os.path.join(tmp_dir, "donut.png")
fig.savefig(donut_path, dpi=200, bbox_inches="tight", transparent=True)
plt.show()

# ── 2. Tier Coverage Bar Chart ──
tier_labels, coverage_values, bar_colors = [], [], []
tier_color_map = {
    "required":    COLORS["tier_required"],
    "preferred":   COLORS["tier_preferred"],
    "nice_to_have": COLORS["tier_nice"],
}
for tier in ["required", "preferred", "nice_to_have"]:
    if tier in breakdown:
        tier_labels.append(tier.replace("_", " ").title())
        coverage_values.append(breakdown[tier]["tier_pct"])
        bar_colors.append(hex_to_float_rgb(tier_color_map.get(tier, COLORS["accent"])))

fig, ax = plt.subplots(figsize=(8, 3))
fig.patch.set_facecolor("white")
y_pos = np.arange(len(tier_labels))
ax.barh(y_pos, [100]*len(tier_labels), height=0.5,
        color=hex_to_float_rgb(COLORS["light_gray"]), zorder=2)
bars = ax.barh(y_pos, coverage_values, height=0.5, color=bar_colors,
               edgecolor="white", linewidth=1.5, zorder=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(tier_labels, fontsize=11, fontweight="bold")
ax.set_xlim(0, 110)
ax.set_xlabel("Coverage (%)", fontsize=10)
ax.set_title("Skills Coverage by Tier", fontsize=14, fontweight="bold", pad=15)
for i, v in enumerate(coverage_values):
    ax.text(v + 2, i, f"{v:.1f}%", va="center", ha="left",
            fontsize=11, fontweight="bold", color=hex_to_float_rgb(COLORS["dark_gray"]))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(left=False)
ax.xaxis.grid(True, alpha=0.3, linestyle="--")
plt.tight_layout()
tier_bar_path = os.path.join(tmp_dir, "tier_bar.png")
fig.savefig(tier_bar_path, dpi=200, bbox_inches="tight")
plt.show()

# ── 3. Matched vs Missing Grouped Bar Chart ──
tiers = ["required", "preferred", "nice_to_have"]
labels = [t.replace("_", " ").title() for t in tiers]
matched = [len(breakdown[t]["matched"]) for t in tiers]
missing = [len(breakdown[t]["missing"]) for t in tiers]
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor("white")
bars1 = ax.bar(x - width/2, matched, width, label="Matched",
               color=hex_to_float_rgb(COLORS["success"]), edgecolor="white", linewidth=1.5, zorder=3)
bars2 = ax.bar(x + width/2, missing, width, label="Missing",
               color=hex_to_float_rgb(COLORS["danger"]), edgecolor="white", linewidth=1.5, zorder=3)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11, fontweight="bold")
ax.set_ylabel("Number of Skills", fontsize=10)
ax.set_title("Matched vs Missing Skills", fontsize=14, fontweight="bold", pad=15)
ax.legend(fontsize=10, loc="upper right")
for bar in bars1:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.1,
                str(int(h)), ha="center", va="bottom", fontweight="bold", fontsize=10)
for bar in bars2:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.1,
                str(int(h)), ha="center", va="bottom", fontweight="bold", fontsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.grid(True, alpha=0.3, linestyle="--")
ax.set_axisbelow(True)
plt.tight_layout()
matched_missing_path = os.path.join(tmp_dir, "matched_missing.png")
fig.savefig(matched_missing_path, dpi=200, bbox_inches="tight")
plt.show()

# ── 4. Skills Radar Chart ──
skills_list = sorted(list(candidate_skills))[:12]
N = len(skills_list)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values = [1.0] * N
angles += angles[:1]
values += values[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
fig.patch.set_facecolor("white")
ax.plot(angles, values, "o-", linewidth=2, color=hex_to_float_rgb(COLORS["accent"]))
ax.fill(angles, values, alpha=0.15, color=hex_to_float_rgb(COLORS["accent"]))
ax.set_xticks(angles[:-1])
ax.set_xticklabels([s.title() for s in skills_list], fontsize=8, fontweight="bold")
ax.set_ylim(0, 1.3)
ax.set_yticklabels([])
ax.grid(color=hex_to_float_rgb(COLORS["mid_gray"]), linestyle="--", alpha=0.5)
ax.set_title("Candidate Skills Overview", fontsize=14, fontweight="bold", pad=20)
plt.tight_layout()
radar_path = os.path.join(tmp_dir, "radar.png")
fig.savefig(radar_path, dpi=200, bbox_inches="tight")
plt.show()

print("\n✅ All 4 charts generated and displayed.")

C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\1634210250.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\1634210250.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\1634210250.py:108: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



✅ All 4 charts generated and displayed.


C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\1634210250.py:131: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [38]:
# ════════════════════════════════════════════════════════════
# BUILD & SAVE PDF REPORT
# ════════════════════════════════════════════════════════════

def sanitize_text(text):
    """Remove non-latin1 characters for fpdf Helvetica compatibility."""
    result = []
    for ch in text:
        try:
            ch.encode('latin-1')
            result.append(ch)
        except UnicodeEncodeError:
            replacements = {
                '\u2014': '-', '\u2013': '-',
                '\u2018': "'", '\u2019': "'",
                '\u201c': '"', '\u201d': '"',
                '\u2022': '*', '\u2026': '...',
            }
            result.append(replacements.get(ch, ''))
    return ''.join(result)


class CareerReportPDF(FPDF):
    def __init__(self, candidate_name="", target_role=""):
        super().__init__("P", "mm", "A4")
        self.candidate_name = candidate_name
        self.target_role = target_role
        self.set_auto_page_break(auto=True, margin=25)

    def header(self):
        if self.page_no() == 1:
            return
        self.set_font("Helvetica", "B", 8)
        r, g, b = hex_to_rgb(COLORS["mid_gray"])
        self.set_text_color(r, g, b)
        self.cell(0, 8, sanitize_text(f"Career Analysis Report - {self.candidate_name}"), align="L")
        self.ln(3)
        r, g, b = hex_to_rgb(COLORS["accent"])
        self.set_draw_color(r, g, b)
        self.set_line_width(0.5)
        self.line(10, 13, 200, 13)
        self.ln(5)

    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        r, g, b = hex_to_rgb(COLORS["mid_gray"])
        self.set_text_color(r, g, b)
        self.cell(0, 10, f"Page {self.page_no()}/{{nb}}", align="C")

    def section_title(self, title):
        self.set_font("Helvetica", "B", 16)
        r, g, b = hex_to_rgb(COLORS["primary"])
        self.set_text_color(r, g, b)
        self.cell(0, 12, sanitize_text(title), ln=True)
        r, g, b = hex_to_rgb(COLORS["accent"])
        self.set_draw_color(r, g, b)
        self.set_line_width(0.8)
        self.line(self.l_margin, self.get_y(), 100, self.get_y())
        self.ln(6)

    def sub_title(self, title):
        self.set_font("Helvetica", "B", 13)
        r, g, b = hex_to_rgb(COLORS["dark_gray"])
        self.set_text_color(r, g, b)
        self.cell(0, 10, sanitize_text(title), ln=True)
        self.ln(2)

    def body_text(self, text, bold=False):
        style = "B" if bold else ""
        self.set_font("Helvetica", style, 10)
        r, g, b = hex_to_rgb(COLORS["dark_gray"])
        self.set_text_color(r, g, b)
        self.multi_cell(0, 6, sanitize_text(text))
        self.ln(2)

    def info_row(self, label, value):
        self.set_font("Helvetica", "B", 10)
        r, g, b = hex_to_rgb(COLORS["primary"])
        self.set_text_color(r, g, b)
        self.cell(50, 7, sanitize_text(label), ln=False)
        self.set_font("Helvetica", "", 10)
        r, g, b = hex_to_rgb(COLORS["dark_gray"])
        self.set_text_color(r, g, b)
        self.cell(0, 7, sanitize_text(value), ln=True)

    def skill_badge(self, skill, is_matched=True):
        color = COLORS["success"] if is_matched else COLORS["danger"]
        icon = "[+]" if is_matched else "[-]"
        self.set_font("Helvetica", "", 9)
        r, g, b = hex_to_rgb(color)
        self.set_text_color(r, g, b)
        self.cell(0, 5, sanitize_text(f"  {icon}  {skill.title()}"), ln=True)


def clean_markdown_text(text):
    text = re_mod.sub(r'\*\*([^*]+)\*\*', r'\\1', text)
    text = re_mod.sub(r'\[([^\]]+)\]\([^)]+\)', r'\\1', text)
    text = text.replace('`', '')
    return text


def parse_roadmap_markdown(roadmap_text):
    lines = roadmap_text.split("\n")
    sections = []
    current_section = None
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped.startswith("### "):
            if current_section:
                sections.append(current_section)
            current_section = {"title": stripped[4:], "subsections": [], "items": []}
        elif stripped.startswith("#### "):
            if current_section:
                current_section["subsections"].append({"title": stripped[5:], "items": []})
        elif stripped.startswith("- "):
            content = stripped[2:]
            if current_section:
                if current_section["subsections"]:
                    current_section["subsections"][-1]["items"].append(content)
                else:
                    current_section["items"].append(content)
    if current_section:
        sections.append(current_section)
    return sections


# ── Use in-memory data (strategist_output from Step 8) ──
data = strategist_output

pdf = CareerReportPDF(
    candidate_name=data["candidate"]["name"],
    target_role=data["target_role"]["title"]
)
pdf.alias_nb_pages()

# ── Page 1: Cover ──
pdf.add_page()
pdf.ln(20)
pdf.set_font("Helvetica", "B", 28)
r, g, b = hex_to_rgb(COLORS["primary"])
pdf.set_text_color(r, g, b)
pdf.cell(0, 15, "Career Analysis Report", align="C", ln=True)
pdf.set_font("Helvetica", "", 12)
r, g, b = hex_to_rgb(COLORS["mid_gray"])
pdf.set_text_color(r, g, b)
pdf.cell(0, 8, "AI-Powered Skills Gap Analysis & Learning Roadmap", align="C", ln=True)
pdf.ln(5)
r, g, b = hex_to_rgb(COLORS["accent"])
pdf.set_draw_color(r, g, b)
pdf.set_line_width(1)
pdf.line(60, pdf.get_y(), 150, pdf.get_y())
pdf.ln(12)

pdf.set_font("Helvetica", "B", 13)
r, g, b = hex_to_rgb(COLORS["dark_gray"])
pdf.set_text_color(r, g, b)
pdf.cell(0, 8, "Candidate Profile", ln=True)
pdf.ln(2)
pdf.info_row("Name:", data["candidate"]["name"])
pdf.info_row("Current Title:", data["candidate"]["title"])
pdf.info_row("Total Skills:", str(len(data["candidate"]["skills_found"])))
pdf.ln(4)
pdf.set_font("Helvetica", "B", 13)
pdf.cell(0, 8, "Target Role", ln=True)
pdf.ln(2)
pdf.info_row("Position:", data["target_role"]["title"])
pdf.info_row("Company:", data["target_role"]["company"])
pdf.ln(8)

if os.path.exists(donut_path):
    img_w = 65
    x = (210 - img_w) / 2
    pdf.image(donut_path, x=x, y=pdf.get_y(), w=img_w)
    pdf.ln(70)

pdf.set_font("Helvetica", "I", 9)
r, g, b = hex_to_rgb(COLORS["mid_gray"])
pdf.set_text_color(r, g, b)
pdf.cell(0, 8, f"Generated on {datetime.now().strftime('%B %d, %Y at %H:%M')}", align="C", ln=True)

# ── Page 2: Skills Breakdown ──
pdf.add_page()
pdf.section_title("Skills Match Breakdown")
overall = data["match_score"]["overall"]
bd = data["match_score"]["breakdown"]
pdf.body_text(f"Overall match score: {overall}%. Below is the detailed breakdown by skill tier.")

if os.path.exists(tier_bar_path):
    pdf.image(tier_bar_path, x=15, w=180)
    pdf.ln(5)
if os.path.exists(matched_missing_path):
    pdf.image(matched_missing_path, x=25, w=155)
    pdf.ln(5)

pdf.sub_title("[+] Matched Skills")
for tier in ["required", "preferred", "nice_to_have"]:
    if tier in bd and bd[tier]["matched"]:
        label = tier.replace("_", " ").title()
        pdf.set_font("Helvetica", "B", 10)
        r, g, b = hex_to_rgb(COLORS["primary"])
        pdf.set_text_color(r, g, b)
        pdf.cell(0, 6, sanitize_text(f"  {label}:"), ln=True)
        for skill in bd[tier]["matched"]:
            pdf.skill_badge(skill, is_matched=True)
        pdf.ln(2)

pdf.sub_title("[-] Missing Skills (Gap)")
for tier in ["required", "preferred", "nice_to_have"]:
    if tier in bd and bd[tier]["missing"]:
        label = tier.replace("_", " ").title()
        pdf.set_font("Helvetica", "B", 10)
        r, g, b = hex_to_rgb(COLORS["primary"])
        pdf.set_text_color(r, g, b)
        pdf.cell(0, 6, sanitize_text(f"  {label}:"), ln=True)
        for skill in bd[tier]["missing"]:
            pdf.skill_badge(skill, is_matched=False)
        pdf.ln(2)

# ── Page 3: Skills Radar ──
pdf.add_page()
pdf.section_title("Candidate Skills Overview")
skills_all = data["candidate"]["skills_found"]
pdf.body_text(f"{data['candidate']['name']} has {len(skills_all)} identified skills across technical, tools, and soft skills categories.")

if os.path.exists(radar_path):
    img_w = 140
    x = (210 - img_w) / 2
    pdf.image(radar_path, x=x, w=img_w)
    pdf.ln(5)

pdf.sub_title("Complete Skills Inventory")
sorted_skills = sorted(skills_all)
col_width = 60
x_start = pdf.l_margin
pdf.set_font("Helvetica", "", 9)
r, g, b = hex_to_rgb(COLORS["dark_gray"])
pdf.set_text_color(r, g, b)
for i, skill in enumerate(sorted_skills):
    col = i % 3
    if col == 0 and i > 0:
        pdf.ln(5)
    x = x_start + col * col_width
    pdf.set_x(x)
    pdf.cell(col_width, 5, f"  * {skill.title()}", ln=False)
pdf.ln(10)

# ── Page 4+: Roadmap ──
pdf.add_page()
pdf.section_title("30-Day Learning Roadmap")
roadmap = data.get("roadmap_markdown", "")
if roadmap:
    sections = parse_roadmap_markdown(roadmap)
    for section in sections:
        title = clean_markdown_text(section["title"])
        if pdf.get_y() > 240:
            pdf.add_page()
        pdf.sub_title(title)
        for item in section["items"]:
            item_clean = clean_markdown_text(item)
            pdf.set_font("Helvetica", "", 10)
            r, g, b = hex_to_rgb(COLORS["dark_gray"])
            pdf.set_text_color(r, g, b)
            pdf.set_x(pdf.l_margin + 5)
            pdf.multi_cell(0, 5, sanitize_text(f"* {item_clean.strip()}"))
            pdf.ln(1)
        for subsection in section["subsections"]:
            if pdf.get_y() > 250:
                pdf.add_page()
            sub_title = clean_markdown_text(subsection["title"])
            pdf.set_font("Helvetica", "B", 11)
            r, g, b = hex_to_rgb(COLORS["accent"])
            pdf.set_text_color(r, g, b)
            pdf.cell(0, 8, sanitize_text(f"  {sub_title}"), ln=True)
            for item in subsection["items"]:
                item_clean = clean_markdown_text(item)
                pdf.set_font("Helvetica", "", 10)
                r, g, b = hex_to_rgb(COLORS["dark_gray"])
                pdf.set_text_color(r, g, b)
                pdf.set_x(pdf.l_margin + 8)
                pdf.multi_cell(0, 5, sanitize_text(f"* {item_clean.strip()}"))
                pdf.ln(1)
        pdf.ln(4)
else:
    pdf.body_text("No roadmap data available.")

# ── Save PDF ──
output_path = "career_analysis_report.pdf"
pdf.output(output_path)

file_size = os.path.getsize(output_path)
print(f"\n📄 PDF saved to: {output_path}")
print(f"   File size:    {file_size:,} bytes ({file_size / 1024:.1f} KB)")
print(f"   Total pages:  {pdf.page_no()}")

# Cleanup temp chart images
import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)

print("\n✅ Career analysis report generated successfully!")

C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\3636392754.py:145: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 15, "Career Analysis Report", align="C", ln=True)
C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\3636392754.py:149: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, "AI-Powered Skills Gap Analysis & Learning Roadmap", align="C", ln=True)
C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\3636392754.py:160: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, "Candidate Profile", ln=True)
C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\3636392754.py:81: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=False use new_x=XPos.RIGHT, new_y=YPos.TOP


📄 PDF saved to: career_analysis_report.pdf
   File size:    304,611 bytes (297.5 KB)
   Total pages:  6

✅ Career analysis report generated successfully!


C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\3636392754.py:247: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=False use new_x=XPos.RIGHT, new_y=YPos.TOP.
  pdf.cell(col_width, 5, f"  * {skill.title()}", ln=False)
C:\Users\HP 1030 G7\AppData\Local\Temp\ipykernel_16572\3636392754.py:276: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, sanitize_text(f"  {sub_title}"), ln=True)
